## Exercises — BioImage Simulation with DeepTrack2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIM-Net/SPAOM-CW20-DeepTrack/blob/main/CW20_exercises.ipynb)
&nbsp;
[Repository](https://github.com/AIM-Net/SPAOM-CW20-DeepTrack)

Open-ended exercises to go with [`CW20_Guigo_SPAOM2026.ipynb`](CW20_Guigo_SPAOM2026.ipynb). Work through the main notebook first: these build directly on it. Each exercise has a collapsible solution, so you can try it and then compare.

Start with Exercise 4. Writing your own scatterer is what you need as soon as the object you want to simulate is not a point, a sphere or an ellipsoid.

### Setup

If you are running on **Google Colab / Kaggle**, uncomment and run the next cell. Locally, install the packages once in your environment (`pip install -e .` from the repository root).

In [ ]:
# !pip install deeptrack deeplay

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import deeptrack as dt

# Needed if you subclass a scatterer (Exercise 4).
from deeptrack import units as u
from deeptrack.backend.units import ConversionTable
from deeptrack.optical.scatterers import VolumeScatterer

np.random.seed(2026)

IMAGE_SIZE = 64
fluorescence_microscope = dt.Fluorescence(
    NA=1.4,
    wavelength=680 * dt.units.nm,
    magnification=60,
    resolution=6.5 * dt.units.um,
    refractive_index_medium=1.518,
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

**Exercise 4. Write your own scatterer.**

Pick a shape that none of the built-in classes covers, and implement it as a `VolumeScatterer`
exactly as `Bacillus` is built in section 3 of the main notebook. A **cube or rectangular box** is the easiest
starting point; a dumbbell, a torus or a cone are all within reach of the same recipe.

You need three pieces:

- a `__conversion_table__` declaring the unit of each property, so sizes can be given in µm;
- an `__init__` that forwards your properties to `super().__init__(...)`;
- a `get(self, *ignore, <your properties>, voxel_size, **kwargs)` returning a **3D float array**.

You do **not** handle `position` or `z` — the optics place the volume. `get()` only draws the object centred on the origin.

Three things to keep in mind:

1. The meshgrid order is `Y, X, Z = np.meshgrid(y, x, z)` — note that `Y` comes first. Getting it wrong builds a transposed object, which is invisible for a cube with equal sides.
2. Size the grid for the **rotated** object. A cube turned 45° spans √2 times its side in plane, √3 in 3D. Too small an extent silently clips the corners.
3. Sharp edges alias badly. Compare with and without the `upsample` parameter.

Then image it: face-on, rotated 45°, and through two different modalities.

<details>
<summary><i>Show solution</i></summary>

```python
class Box(VolumeScatterer):
    """A rectangular box. `side` is a scalar for a cube, or three values for a box."""

    __conversion_table__ = ConversionTable(
        side=(u.meter, u.meter),
        rotation=(u.radian, u.radian),
    )

    def __init__(self, side=1e-6, rotation=0, **kwargs):
        super().__init__(side=side, rotation=rotation, **kwargs)

    def _process_properties(self, properties):
        properties = super()._process_properties(properties)
        side = np.ravel(properties["side"]).astype(float)
        properties["side"] = tuple(side) if side.size == 3 else (float(side[0]),) * 3
        properties["rotation"] = float(np.ravel(properties["rotation"])[0])
        return properties

    def get(self, *ignore, side, rotation, voxel_size, **kwargs):
        half_x, half_y, half_z = (s / 2 for s in side)

        # Room for the rotated object: a box spans sqrt(3) x its longest side.
        extent = int(np.ceil(np.sqrt(3) * max(side) / 2 / np.min(voxel_size))) + 1
        grid = [np.arange(-extent, extent + 1) * v for v in voxel_size]
        Y, X, Z = np.meshgrid(grid[1], grid[0], grid[2])   # note the order

        # Rotate the grid, then test against an axis-aligned box.
        XR = np.cos(rotation) * X + np.sin(rotation) * Y
        YR = -np.sin(rotation) * X + np.cos(rotation) * Y

        inside = (np.abs(XR) < half_x) & (np.abs(YR) < half_y) & (np.abs(Z) < half_z)
        return inside.astype(float)


fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (rotation, upsample, title) in zip(axes, [
    (0, 1, "face-on"),
    (np.pi / 4, 1, "rotated 45 deg"),
    (np.pi / 4, 4, "rotated, upsample=4"),
]):
    box = Box(side=1.5 * dt.units.um, rotation=rotation, position=(32, 32),
              intensity=3, upsample=upsample)
    ax.imshow(np.asarray(fluorescence_microscope(box).resolve())[..., 0], cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# Your code here.


**Exercise 5. Simulate your own experiment.**

Take a microscope you actually use and an object you actually image, and build the pipeline end to end:

1. **Your optics.** Put in your own `NA`, `magnification`, emission `wavelength`, and the pixel
   size of your camera as `resolution`. Print the pixel size in the sample
   (`resolution / magnification`) and the PSF width (`0.51 * wavelength / NA`), and check you get
   two to three pixels across it. If DeepTrack2 warns about the sampling, your pixels are too
   large for that objective.
2. **Your sample.** Pick the built-in scatterer closest to your object, give its size in physical
   units, and randomize what varies in reality: position, size, brightness.
3. **Your camera.** Add a background and shot noise with `>> dt.Add(...) >> dt.Poisson(...)`.
4. **Your labels.** Get the positions with `dt.TakeProperties` and a mask with
   `dt.SampleToMasks`, and plot them next to the image.

Does the result look like your data?

<details>
<summary><i>Show solution</i></summary>

An example for a 40x/0.95 air objective, an sCMOS camera and GFP-labelled beads.

```python
# --- 1. Describe your microscope ------------------------------------------
MAGNIFICATION = 40
CAMERA_PIXEL_UM = 6.5          # sCMOS
NA = 0.95                      # 40x air objective
WAVELENGTH_NM = 510            # GFP emission
FIELD = 128                    # pixels

pixel_nm = CAMERA_PIXEL_UM / MAGNIFICATION * 1000
fwhm_nm = 0.51 * WAVELENGTH_NM / NA
print(f"pixel in the sample : {pixel_nm:.0f} nm")
print(f"PSF FWHM            : {fwhm_nm:.0f} nm = {fwhm_nm / pixel_nm:.1f} px")

my_microscope = dt.Fluorescence(
    NA=NA,
    wavelength=WAVELENGTH_NM * dt.units.nm,
    magnification=MAGNIFICATION,
    resolution=CAMERA_PIXEL_UM * dt.units.um,
    output_region=(0, 0, FIELD, FIELD),
)

# --- 2. Describe your sample ----------------------------------------------
bead = dt.Sphere(
    position=lambda: np.random.uniform(8, FIELD - 8, size=2),
    radius=lambda: np.random.uniform(0.4, 0.8) * dt.units.um,
    intensity=lambda: np.random.uniform(0.4, 1.0),
)
sample = bead ^ (lambda: np.random.randint(10, 25))

# --- 3. Image it, with the noise your camera would add --------------------
pipeline = (
    my_microscope(sample)
    >> dt.Add(0.1)
    >> dt.Poisson(snr=20, background=0.1)
)

# --- 4. The ground truth comes along for free -----------------------------
mask_feature = sample >> dt.SampleToMasks(
    lambda: (lambda volume: np.any(volume > 0, axis=-1, keepdims=True)),
    output_region=my_microscope.output_region,
    merge_method="or",
)

image, *positions = (pipeline & dt.TakeProperties(sample, "position")).update()()
mask = np.asarray(mask_feature.resolve())[..., 0]
positions = np.array(positions)
print(f"simulated {len(positions)} beads")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.squeeze(image), cmap="gray")
axes[0].set_title("Simulated image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Ground-truth mask")
axes[2].imshow(np.squeeze(image), cmap="gray")
axes[2].scatter(positions[:, 1], positions[:, 0], s=60, marker="+", c="r")
axes[2].set_title("Positions")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# Your code here.


**Exercise 6. Which simulation choices actually matter?**

Pick one aspect of any simulation in the main notebook and vary it over a wide range, then describe what changes in the resulting images:

| | Try |
|---|---|
| Density | 2 objects per image vs. 60 |
| Noise | `snr` from 2 to 50, with and without a background |
| Optics | `NA` from 0.3 to 1.4, or a 10x vs 100x magnification |
| Shape | perfectly round objects vs. strongly elongated and deformed ones |
| Aberration | `coefficient` from 0 to 4 |

If you trained a network only on your images, what kind of real image would break it?

<details>
<summary><i>Show solution</i></summary>

```python
particle = dt.PointParticle(
    position=lambda: np.random.uniform(5, IMAGE_SIZE - 5, size=2),
    intensity=40,
)
emitters = particle ^ 20
clean = fluorescence_microscope(emitters)

snr_values = [2, 5, 15, 50]
fig, axes = plt.subplots(1, len(snr_values), figsize=(14, 3.6))
for ax, snr in zip(axes, snr_values):
    noisy = clean >> dt.Add(0.1) >> dt.Poisson(snr=snr, background=0.1)
    image = np.asarray(noisy.resolve())[..., 0]

    # How far above the background noise do the emitters stand?
    background = image[:12, :12]
    contrast = (image.max() - background.mean()) / background.std()

    ax.imshow(image, cmap="gray")
    ax.set_title(f"snr = {snr}\npeak / background sigma = {contrast:.0f}")
    ax.axis("off")
plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# Your code here.
